# Airflow — Idempotency, Backfill, Sensors, Dynamic DAGs, TaskFlow

This notebook demonstrates advanced Apache Airflow patterns against a local stack:

- **Idempotency**
- **Backfill**
- **Sensors**
- **Dynamic DAG generation**
- **TaskFlow API**

## Mental Model

Airflow is not just a scheduler. It is an orchestration control plane for **re-runnable**, **observable**, and **recoverable** workflows. In production data engineering, the most important property is usually **idempotency**: if a DAG runs twice, it should not corrupt state or create duplicates.

This notebook uses a realistic telemetry narrative:

- PostgreSQL holds endpoint, metrics, and alerts data.
- Airflow runs locally at `http://localhost:8082`.
- Citi-like monitoring pipelines analyze thousands of endpoints and escalate alerts by severity.
- DAGs are created programmatically into the local Airflow DAGs folder, then triggered through the Airflow REST API.

> Idempotency is the most important Airflow property for data engineering.  
> A DAG that can be re-run safely is a DAG that can be monitored, recovered, and backfilled.  
> SLA-driven pipelines depend on this.

In [ ]:
import json
import os
import time
import textwrap
from datetime import datetime, timedelta, timezone
from pathlib import Path

import requests
import psycopg2
from psycopg2.extras import RealDictCursor


AIRFLOW_BASE_URL = "http://localhost:8082/api/v1"
AIRFLOW_USERNAME = "admin"
AIRFLOW_PASSWORD = "admin"

POSTGRES_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "de_telemetry",
    "user": "de_admin",
    "password": "DeAdmin2026!",
}

TECH_STACK = {
    "kafka": {"host": "localhost:9092", "image": "confluentinc/cp-kafka:7.6.0", "container": "citi_kafka"},
    "spark": {"version": "pyspark==3.5.4", "master": "local[*]", "java_home": r"C:/Program Files/Java/jre1.8.0_481", "hadoop_home": r"C:/hadoop"},
    "airflow": {"url": "http://localhost:8082", "version": "apache/airflow:2.8.0", "executor": "LocalExecutor"},
    "mlflow": {"url": "http://localhost:5000", "backend": "SQLite"},
    "dbt": {"exe": r"C:/py_venv/proj_educate/Scripts/dbt.exe", "profiles": "~/.dbt/profiles.yml", "project": "citi_dbt", "target": "postgres"},
    "databricks": {"host": "https://dbc-9f35a83d-b4e7.cloud.databricks.com", "warehouse_id": "b6657f31d1e7a179"},
    "gcp": {"project": "citi-de-learning", "key": r"D:/Workspace/Technologies/_setup/gcp_key.json"},
    "azure": {"subscription": "b3811436-61fc-4a3a-a6a9-deb05955076d", "az_cli": r"C:\Program Files (x86)\Microsoft SDKs\Azure\CLI2\wbin\az.cmd"},
    "aws": {"profile": "study", "region": "us-east-1", "account": "357811130281"},
}

DATASET_CONTEXT = {
    "endpoints": "10,000 rows | endpoint_id (int PK), name (varchar), region (varchar), status (varchar), category (varchar)",
    "metrics": "500,000 rows | endpoint_id (int FK), metric_name (varchar), value (float), timestamp (timestamptz)",
    "alerts": "25,000 rows | alert_id (int PK), endpoint_id (int FK), severity (varchar), message (text), created_at (timestamptz)",
    "narrative": "6,000+ API endpoints monitored for latency, error rate, throughput; alerts escalate through severity tiers",
}


def get_pg_connection():
    return psycopg2.connect(**POSTGRES_CONFIG)


class AirflowClient:
    def __init__(self, base_url=AIRFLOW_BASE_URL, username=AIRFLOW_USERNAME, password=AIRFLOW_PASSWORD):
        self.base_url = base_url.rstrip("/")
        self.session = requests.Session()
        self.session.auth = (username, password)
        self.session.headers.update({"Content-Type": "application/json"})

    def _url(self, path: str) -> str:
        return f"{self.base_url}/{path.lstrip('/')}"

    def health(self):
        response = self.session.get(self._url("../health"), timeout=15)
        response.raise_for_status()
        return response.json()

    def get_dag(self, dag_id: str):
        response = self.session.get(self._url(f"dags/{dag_id}"), timeout=15)
        response.raise_for_status()
        return response.json()

    def unpause_dag(self, dag_id: str):
        response = self.session.patch(
            self._url(f"dags/{dag_id}"),
            data=json.dumps({"is_paused": False}),
            timeout=15,
        )
        response.raise_for_status()
        return response.json()

    def trigger_dag(self, dag_id: str, logical_date: str | None = None, conf: dict | None = None, dag_run_id: str | None = None):
        payload = {}
        if logical_date is not None:
            payload["logical_date"] = logical_date
        if conf is not None:
            payload["conf"] = conf
        if dag_run_id is not None:
            payload["dag_run_id"] = dag_run_id

        response = self.session.post(
            self._url(f"dags/{dag_id}/dagRuns"),
            data=json.dumps(payload),
            timeout=20,
        )
        response.raise_for_status()
        return response.json()

    def get_dag_run(self, dag_id: str, dag_run_id: str):
        response = self.session.get(self._url(f"dags/{dag_id}/dagRuns/{dag_run_id}"), timeout=15)
        response.raise_for_status()
        return response.json()

    def wait_for_dag_run(self, dag_id: str, dag_run_id: str, timeout_seconds: int = 180, poll_seconds: int = 3):
        deadline = time.time() + timeout_seconds
        history = []

        while time.time() < deadline:
            dag_run = self.get_dag_run(dag_id, dag_run_id)
            state = dag_run["state"]
            history.append((datetime.now().isoformat(timespec="seconds"), state))
            if state in {"success", "failed"}:
                return dag_run, history
            time.sleep(poll_seconds)

        raise TimeoutError(f"DAG run {dag_run_id} for {dag_id} did not finish within {timeout_seconds} seconds.")


def infer_airflow_dags_folder() -> Path:
    candidates = [
        Path(os.environ.get("AIRFLOW_HOME", "")) / "dags" if os.environ.get("AIRFLOW_HOME") else None,
        Path.home() / "airflow" / "dags",
        Path(r"D:/Workspace/Technologies/airflow/dags"),
        Path(r"C:/airflow/dags"),
    ]
    for path in candidates:
        if path and path.exists():
            return path
    raise FileNotFoundError("Could not infer the local Airflow DAGs folder. Set AIRFLOW_HOME or create a standard dags directory.")


def write_dag_file(filename: str, content: str) -> Path:
    dags_folder = infer_airflow_dags_folder()
    dags_folder.mkdir(parents=True, exist_ok=True)
    path = dags_folder / filename
    path.write_text(textwrap.dedent(content).strip() + "\n", encoding="utf-8")
    return path


airflow = AirflowClient()
print("Airflow health:", airflow.health())
print("Dataset context loaded for endpoints / metrics / alerts.")

## 1) Idempotency Pattern

We will write a DAG that inserts a deterministic audit row into PostgreSQL using `INSERT ... ON CONFLICT DO NOTHING`, trigger it twice, and verify that the row count does not increase on re-run.

In [ ]:
IDEMPOTENT_DAG_ID = "citi_idempotent_endpoint_audit"
IDEMPOTENT_AUDIT_TABLE = "airflow_idempotent_audit"

with get_pg_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(f"""
        CREATE TABLE IF NOT EXISTS {IDEMPOTENT_AUDIT_TABLE} (
            run_key text PRIMARY KEY,
            note text NOT NULL,
            created_at timestamptz NOT NULL DEFAULT now()
        )
        """)
    conn.commit()

idempotent_dag_code = f"""
from datetime import datetime
from airflow import DAG
from airflow.providers.postgres.operators.postgres import PostgresOperator

with DAG(
    dag_id="{IDEMPOTENT_DAG_ID}",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["citi", "idempotency", "postgres"],
) as dag:
    idempotent_insert = PostgresOperator(
        task_id="idempotent_insert",
        postgres_conn_id="postgres_default",
        sql=\"\"\"
        CREATE TABLE IF NOT EXISTS {IDEMPOTENT_AUDIT_TABLE} (
            run_key text PRIMARY KEY,
            note text NOT NULL,
            created_at timestamptz NOT NULL DEFAULT now()
        );

        INSERT INTO {IDEMPOTENT_AUDIT_TABLE} (run_key, note)
        VALUES ('endpoint-monitor-bootstrap', 'Inserted once even if DAG is retriggered')
        ON CONFLICT (run_key) DO NOTHING;
        \"\"\",
    )
"""

dag_path = write_dag_file(f"{IDEMPOTENT_DAG_ID}.py", idempotent_dag_code)
print("Wrote DAG:", dag_path)

time.sleep(8)
airflow.unpause_dag(IDEMPOTENT_DAG_ID)

run_1 = airflow.trigger_dag(IDEMPOTENT_DAG_ID, dag_run_id=f"manual__{int(time.time())}__1")
dag_run_id_1 = run_1["dag_run_id"]
result_1, history_1 = airflow.wait_for_dag_run(IDEMPOTENT_DAG_ID, dag_run_id_1)

run_2 = airflow.trigger_dag(IDEMPOTENT_DAG_ID, dag_run_id=f"manual__{int(time.time())}__2")
dag_run_id_2 = run_2["dag_run_id"]
result_2, history_2 = airflow.wait_for_dag_run(IDEMPOTENT_DAG_ID, dag_run_id_2)

with get_pg_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(f"SELECT COUNT(*) FROM {IDEMPOTENT_AUDIT_TABLE}")
        audit_count = cur.fetchone()[0]

print("Run 1 state:", result_1["state"])
print("Run 2 state:", result_2["state"])
print(f"Idempotency verified: {audit_count} rows, no duplicates")

## 2) Backfill Simulation

Now we trigger the same DAG with a `logical_date` set 7 days in the past to simulate a controlled backfill.

In [ ]:
backfill_logical_date = (datetime.now(timezone.utc) - timedelta(days=7)).replace(microsecond=0).isoformat()
backfill_run_id = f"backfill__{int(time.time())}"

backfill_run = airflow.trigger_dag(
    IDEMPOTENT_DAG_ID,
    logical_date=backfill_logical_date,
    dag_run_id=backfill_run_id,
    conf={"reason": "backfill demonstration", "scope": "7_days_ago"},
)

backfill_result, backfill_history = airflow.wait_for_dag_run(IDEMPOTENT_DAG_ID, backfill_run_id)

execution_summary = {
    "dag_id": IDEMPOTENT_DAG_ID,
    "dag_run_id": backfill_result["dag_run_id"],
    "state": backfill_result["state"],
    "logical_date": backfill_result["logical_date"],
    "start_date": backfill_result.get("start_date"),
    "end_date": backfill_result.get("end_date"),
    "history": backfill_history[-5:],
}

print("Backfill execution summary:")
print(json.dumps(execution_summary, indent=2))

## 3) Sensor Pattern

We create a `FileSensor` DAG that waits for a file to appear. The notebook creates that file after a short delay, then polls the DAG run until completion.

In [ ]:
SENSOR_DAG_ID = "citi_file_sensor_demo"
SENSOR_FILE_PATH = Path.cwd() / "sensor_drop" / "go_signal.txt"
SENSOR_FILE_PATH.parent.mkdir(parents=True, exist_ok=True)

sensor_dag_code = f"""
from datetime import datetime
from airflow import DAG
from airflow.operators.empty import EmptyOperator
from airflow.sensors.filesystem import FileSensor

with DAG(
    dag_id="{SENSOR_DAG_ID}",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["citi", "sensor", "filesystem"],
) as dag:
    wait_for_file = FileSensor(
        task_id="wait_for_file",
        filepath=r"{SENSOR_FILE_PATH.as_posix()}",
        fs_conn_id="fs_default",
        poke_interval=1,
        timeout=60,
        mode="poke",
    )

    done = EmptyOperator(task_id="done")

    wait_for_file >> done
"""

sensor_path = write_dag_file(f"{SENSOR_DAG_ID}.py", sensor_dag_code)
print("Wrote DAG:", sensor_path)

if SENSOR_FILE_PATH.exists():
    SENSOR_FILE_PATH.unlink()

time.sleep(8)
airflow.unpause_dag(SENSOR_DAG_ID)

sensor_run = airflow.trigger_dag(SENSOR_DAG_ID, dag_run_id=f"sensor__{int(time.time())}")
sensor_run_id = sensor_run["dag_run_id"]

time.sleep(3)
SENSOR_FILE_PATH.write_text("ready\n", encoding="utf-8")

sensor_result, sensor_history = airflow.wait_for_dag_run(SENSOR_DAG_ID, sensor_run_id, timeout_seconds=120, poll_seconds=2)

print("Sensor run state:", sensor_result["state"])
print("FileSensor triggered and completed")

## 4) Dynamic DAG

This DAG generates one task per Citi region: `NYC1`, `SNG1`, `LDN1`, `TKY1`, `SYD1`.  
Each task queries PostgreSQL for alert counts by region.

In [ ]:
DYNAMIC_DAG_ID = "citi_dynamic_alerts_by_region"
REGIONS = ["NYC1", "SNG1", "LDN1", "TKY1", "SYD1"]

dynamic_dag_code = f"""
from datetime import datetime
from airflow import DAG
from airflow.operators.python import PythonOperator
import psycopg2

POSTGRES_CONFIG = {POSTGRES_CONFIG!r}
REGIONS = {REGIONS!r}

def query_alert_count(region: str, **_context):
    conn = psycopg2.connect(**POSTGRES_CONFIG)
    try:
        with conn.cursor() as cur:
            cur.execute(
                \"""
                SELECT %s AS region, COUNT(*) AS alert_count
                FROM alerts a
                JOIN endpoints e
                  ON a.endpoint_id = e.endpoint_id
                WHERE e.region = %s
                \""",
                (region, region),
            )
            row = cur.fetchone()
            print({{"region": row[0], "alert_count": row[1]}})
    finally:
        conn.close()

with DAG(
    dag_id="{DYNAMIC_DAG_ID}",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["citi", "dynamic-dag", "alerts"],
) as dag:
    previous = None
    for region in REGIONS:
        task = PythonOperator(
            task_id=f"alert_count_{{region.lower()}}",
            python_callable=query_alert_count,
            op_kwargs={{"region": region}},
        )
        if previous:
            previous >> task
        previous = task
"""

dynamic_path = write_dag_file(f"{DYNAMIC_DAG_ID}.py", dynamic_dag_code)
print("Wrote DAG:", dynamic_path)

time.sleep(8)
airflow.unpause_dag(DYNAMIC_DAG_ID)

dynamic_run = airflow.trigger_dag(DYNAMIC_DAG_ID, dag_run_id=f"dynamic__{int(time.time())}")
dynamic_result, dynamic_history = airflow.wait_for_dag_run(DYNAMIC_DAG_ID, dynamic_run["dag_run_id"], timeout_seconds=180)

region_results = []
with get_pg_connection() as conn:
    with conn.cursor(cursor_factory=RealDictCursor) as cur:
        cur.execute("""
        SELECT e.region, COUNT(*) AS alert_count
        FROM alerts a
        JOIN endpoints e
          ON a.endpoint_id = e.endpoint_id
        WHERE e.region = ANY(%s)
        GROUP BY e.region
        ORDER BY e.region
        """, (REGIONS,))
        region_results = cur.fetchall()

print("Dynamic DAG state:", dynamic_result["state"])
print("Region alert counts:")
for row in region_results:
    print(dict(row))

## 5) TaskFlow API

Next we rewrite the pattern with `@dag` and `@task`. This makes task composition simpler and XCom passing implicit.

In [ ]:
TASKFLOW_DAG_ID = "citi_taskflow_alerts_by_region"

taskflow_dag_code = f"""
from datetime import datetime
from airflow.decorators import dag, task
import psycopg2

POSTGRES_CONFIG = {POSTGRES_CONFIG!r}
REGIONS = {REGIONS!r}

@dag(
    dag_id="{TASKFLOW_DAG_ID}",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["citi", "taskflow", "alerts"],
)
def taskflow_alerts_by_region():
    @task
    def get_regions():
        return REGIONS

    @task
    def query_region_alerts(region: str):
        conn = psycopg2.connect(**POSTGRES_CONFIG)
        try:
            with conn.cursor() as cur:
                cur.execute(
                    \"""
                    SELECT COUNT(*)
                    FROM alerts a
                    JOIN endpoints e
                      ON a.endpoint_id = e.endpoint_id
                    WHERE e.region = %s
                    \""",
                    (region,),
                )
                count = cur.fetchone()[0]
                result = {{"region": region, "alert_count": count}}
                print(result)
                return result
        finally:
            conn.close()

    @task
    def summarize(results: list[dict]):
        total_alerts = sum(item["alert_count"] for item in results)
        summary = {{"regions": len(results), "total_alerts": total_alerts, "details": results}}
        print(summary)
        return summary

    summaries = query_region_alerts.expand(region=get_regions())
    summarize(summaries)

taskflow_alerts_by_region()
"""

taskflow_path = write_dag_file(f"{TASKFLOW_DAG_ID}.py", taskflow_dag_code)
print("Wrote DAG:", taskflow_path)

time.sleep(8)
airflow.unpause_dag(TASKFLOW_DAG_ID)

taskflow_run = airflow.trigger_dag(TASKFLOW_DAG_ID, dag_run_id=f"taskflow__{int(time.time())}")
taskflow_result, taskflow_history = airflow.wait_for_dag_run(TASKFLOW_DAG_ID, taskflow_run["dag_run_id"], timeout_seconds=180)

print("TaskFlow DAG state:", taskflow_result["state"])
print("TaskFlow uses implicit XCom passing between decorated tasks.")
print("Traditional PythonOperator requires more wiring; TaskFlow is cleaner for Python-native DAG authoring.")

## 6) What Just Happened

You just exercised five advanced Airflow ideas against a local engineering stack:

1. **Idempotency**  
   A DAG inserted the same logical record twice without duplication by using `ON CONFLICT DO NOTHING`.

2. **Backfill**  
   A run was triggered with an older `logical_date`, simulating controlled recovery for missed periods.

3. **Sensors**  
   A `FileSensor` blocked execution until an external condition became true.

4. **Dynamic DAG generation**  
   Tasks were created programmatically from a list of Citi regions.

5. **TaskFlow API**  
   The same logic became more compact and expressive with decorators and implicit XCom behavior.

### Production Takeaway

Idempotency is the most important Airflow property for data engineering.  
A DAG that can be re-run safely is a DAG that can be monitored, recovered, and backfilled.  
Citi's SLA-driven pipelines depend on this.